# 02 — LightGBM WFO (v4): True-Alpha Window + Fee Impact

**Extends:** `02_lgbm_omni_0fee_v3.ipynb`  
**Artifacts:** `artifacts/02_lgbm_omni_0fee_v4/`  

## Research questions

1. **True alpha test:** OOS starts 2024-11-10 when BTC ≈ \$73k — same as today.  
   B\&H ≈ 0%, so any positive strategy return is pure structural alpha.

2. **Fee survival:** Which schemes clear the 0.05% taker fee hurdle after including
   real execution costs (Spot long: maker entry = 0%, taker time-exit = 0.05%)?

## Key differences vs v3

| Parameter | v3 | v4 |
|-----------|-----|-----|
| OOS start | 2024-01-01 | **2024-11-10** (B\&H ≈ 0%) |
| Signal threshold | 0.75 (global) | **Per-scheme** (0.60 / 0.70 / 0.75 / 0.80) |
| Fee model | Zero-fee only | **Zero-fee + fee-adjusted side-by-side** |
| Equity curve | Step-function | **MTM linear interpolation** |
| Prob panel | Raw series | **Raw + rolling mean overlay** |
| Extra figures | — | **Monthly PnL heatmap, fee-drag comparison** |

## Fee model (Spot Longs only — all current signals are long)

```
Entry  : limit order → MAKER_FEE = 0.00%
Exit   : time exit   → SPOT_TAKER_FEE = 0.05%
Round-trip cost: 0.05% per trade
Funding: zero (Spot; no overnight funding)
```

Short signals (if added later) → Futures, funding RECEIVED = +0.00077%/h.

In [ ]:
import json
import sys
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Fixed Horizon target (V2 winner) ──────────────────────────────────────────
FH_HORIZON   = 72       # 6h at 5m
FH_THRESHOLD = 0.003    # >0.3% forward return

# ── Best LGBM params — locked from V2 fixed_horizon grid winner ───────────────
BEST_PARAMS = {
    'num_leaves':       15,
    'max_depth':        8,
    'learning_rate':    0.05,
    'colsample_bytree': 0.5,
}
BASE_LGB_PARAMS = dict(
    objective         = 'binary',
    metric            = 'auc',
    verbose           = -1,
    n_jobs            = -1,
    seed              = 42,
    n_estimators      = 300,
    min_child_samples = 50,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
)
WFO_PARAMS = {**BASE_LGB_PARAMS, **BEST_PARAMS}

# ── WFO settings ──────────────────────────────────────────────────────────────
EMBARGO_BARS   = FH_HORIZON
EMBARGO_DUR    = pd.Timedelta(minutes=5 * EMBARGO_BARS)
MIN_TRAIN_BARS = 5_000

# True-alpha window: BTC at ~73k on 2024-11-10, same as today → B&H ≈ 0%
OOS_START = pd.Timestamp('2024-11-10', tz='UTC')

# ── Fee model (Spot Longs — all current signals are long) ────────────────────
MAKER_FEE         = 0.0000    # 0%    — limit entries
SPOT_TAKER_FEE    = 0.0005   # 0.05% — SL / time exits on spot longs
FUTURES_TAKER_FEE = 0.0005   # 0.05% — reserved for future short signals
BUFFER            = 0.0005   # 5bp penetration buffer (reserved for limit order modeling)
SPOT_FUNDING_H    = 0.0      # 0% — spot longs: no funding
SHORT_FUNDING_H   = 0.0000077  # +0.00077%/h RECEIVED on futures shorts

ROUND_TRIP_FEE = MAKER_FEE + SPOT_TAKER_FEE  # = 0.05% per trade

# ── Per-scheme optimal thresholds (from v3 sensitivity analysis) ──────────────
# Selected to maximise EV while keeping ≥10 trades in the ~18-month OOS window
SCHEME_THRESHOLDS = {
    'expanding':  0.60,   # p>0.60 → EV=+0.498%, 26 trades (v3 full OOS)
    'long_2yr':   0.70,   # p>0.70 → EV=+0.615%, 69 trades
    'medium_1yr': 0.75,   # p>0.75 → EV=+0.461%, 117 trades
    'short_3mo':  0.80,   # p>0.80 → EV=+0.175%, 422 trades
}

print(f'Fixed Horizon  : {FH_HORIZON} bars ({FH_HORIZON*5}min)  threshold={FH_THRESHOLD:.1%}')
print(f'OOS start      : {OOS_START.date()}  ← BTC ≈ $73k → B&H ≈ 0%')
print(f'Round-trip fee : {ROUND_TRIP_FEE:.2%}  (maker 0% + taker {SPOT_TAKER_FEE:.2%})')
print(f'Per-scheme thresholds: {SCHEME_THRESHOLDS}')

In [ ]:
import importlib

def find_repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / 'pyproject.toml').exists():
            return p
        p = p.parent
    raise RuntimeError('pyproject.toml not found')

REPO_ROOT    = find_repo_root()
RAW_DIR      = REPO_ROOT / 'data' / 'raw'
FEATURES_DIR = REPO_ROOT / 'data' / 'features'
CACHE_DIR    = REPO_ROOT / 'data' / 'cache'

ARTIFACTS_DIR = REPO_ROOT / 'artifacts' / '02_lgbm_omni_0fee_v4'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Prob cache: avoid re-running 72 fits if kernel restarts
PROB_CACHE_DIR = CACHE_DIR / '02_lgbm_omni_0fee_v4_probs'
PROB_CACHE_DIR.mkdir(parents=True, exist_ok=True)

OHLCV_5M_PATH  = RAW_DIR      / 'BTCUSDT_5m.parquet'
FEATURES_CLEAN = FEATURES_DIR / 'BTCUSDT_5m_structural_clean.parquet'
REGISTRY_PATH  = FEATURES_DIR / 'feature_registry_v2.json'

# Always reload from disk — avoids stale-module errors
sys.path.insert(0, str(REPO_ROOT / 'src'))
import hmats.evaluation.reporting as _reporting_mod
importlib.reload(_reporting_mod)
from hmats.evaluation.reporting import (
    generate_tearsheet,
    generate_fee_comparison,
    _equity_from_trades,
)

print(f'REPO_ROOT     : {REPO_ROOT}')
print(f'ARTIFACTS_DIR : {ARTIFACTS_DIR}')
print(f'PROB_CACHE    : {PROB_CACHE_DIR}')
print(f'Clean feats   : {FEATURES_CLEAN.exists()}')

## Data Loading & Feature Assembly

In [ ]:
print('Loading OHLCV (5m) ...')
df_raw = pd.read_parquet(OHLCV_5M_PATH,
                         columns=['open', 'high', 'low', 'close', 'volume'])
if df_raw.index.tz is None:
    df_raw.index = df_raw.index.tz_localize('UTC')

print('Loading clean structural features (V1 patch applied) ...')
df_struct = pd.read_parquet(FEATURES_CLEAN)
with open(REGISTRY_PATH) as f:
    registry = json.load(f)
STRUCT_FEATURES = registry['ablation_subsets']['all']

print(f'  5m bars  : {len(df_raw):,}  {df_raw.index[0].date()} -> {df_raw.index[-1].date()}')
print(f'  Features : {len(STRUCT_FEATURES)} structural')

In [ ]:
def build_momentum_features(df5m: pd.DataFrame) -> pd.DataFrame:
    """4 safe momentum features, all shift(1)-guarded."""
    close  = df5m['close']
    volume = df5m['volume']
    feats  = pd.DataFrame(index=df5m.index, dtype='float32')
    for n_bars, tag in [(3, '15m'), (6, '30m'), (12, '1h')]:
        feats[f'mom_ret_{tag}'] = close.pct_change(n_bars).shift(1).astype('float32')
    vol_diff  = volume.diff(1)
    vol_std   = volume.rolling(72, min_periods=36).std() + 1e-8
    vol_slope = vol_diff.rolling(5, min_periods=3).mean() / vol_std
    feats['mom_vol_slope_5bar'] = vol_slope.shift(1).clip(-5, 5).astype('float32')
    return feats

df_mom = build_momentum_features(df_raw)
MOM_FEATURES = df_mom.columns.tolist()
print(f'Momentum features: {MOM_FEATURES}')

In [ ]:
df_base = df_raw.join(df_struct, how='inner').join(df_mom, how='inner')
df_base.dropna(subset=STRUCT_FEATURES, inplace=True)

fwd_close           = df_base['close'].shift(-FH_HORIZON)
df_base['label_fh'] = ((fwd_close / df_base['close'] - 1) > FH_THRESHOLD).astype('int8')
df_base             = df_base.dropna(subset=['label_fh'])

ALL_FEATURES = STRUCT_FEATURES + MOM_FEATURES

print(f'Dataset   : {len(df_base):,} bars  '
      f'{df_base.index[0].date()} -> {df_base.index[-1].date()}')
print(f'Features  : {len(ALL_FEATURES)} ({len(STRUCT_FEATURES)} struct + {len(MOM_FEATURES)} mom)')
print(f'Label pos : {df_base["label_fh"].mean():.3f}  '
      f'({int(df_base["label_fh"].sum()):,} positive bars)')

## WFO Engine

OOS period: **2024-11-10 → 2026-05-27** ≈ 18 monthly windows × 4 schemes ≈ **72 fits** (~2 min).

Probs cached to disk — re-running the cell after a kernel restart skips the fits.

In [ ]:
OOS_END = df_base.index[-1]

WFO_SCHEMES = {
    'expanding':  {'type': 'expanding', 'months': None, 'label': 'Expanding (all history)'},
    'long_2yr':   {'type': 'sliding',   'months': 24,   'label': '2-Year Sliding'},
    'medium_1yr': {'type': 'sliding',   'months': 12,   'label': '1-Year Sliding'},
    'short_3mo':  {'type': 'sliding',   'months': 3,    'label': '3-Month Sliding'},
}

def make_monthly_windows(start, end):
    windows = []
    cur = start
    while cur < end:
        nxt = cur + pd.DateOffset(months=1)
        windows.append((cur, min(nxt, end)))
        cur = nxt
    return windows

windows = make_monthly_windows(OOS_START, OOS_END)
print(f'OOS period      : {OOS_START.date()} -> {OOS_END.date()}')
print(f'Monthly windows : {len(windows)}')
print(f'Total fits      : {len(windows) * len(WFO_SCHEMES)}')

In [ ]:
def run_wfo(scheme_key: str, scheme: dict) -> pd.Series:
    """
    Walk-forward loop for one training-window scheme.
    Results cached to PROB_CACHE_DIR/<scheme_key>.parquet.
    """
    cache_path = PROB_CACHE_DIR / f'{scheme_key}.parquet'
    if cache_path.exists():
        probs = pd.read_parquet(cache_path)['prob']
        print(f'  [{scheme_key}] Loaded from cache: {len(probs):,} predictions')
        return probs

    all_probs: dict = {}
    skipped   = 0
    t0        = time.perf_counter()

    for i, (test_start, test_end) in enumerate(windows):
        cutoff = test_start - EMBARGO_DUR

        if scheme['type'] == 'expanding':
            df_train = df_base[df_base.index < cutoff]
        else:
            win_start = cutoff - pd.DateOffset(months=scheme['months'])
            df_train  = df_base[
                (df_base.index >= win_start) & (df_base.index < cutoff)
            ]

        df_test = df_base[
            (df_base.index >= test_start) & (df_base.index < test_end)
        ]

        if len(df_train) < MIN_TRAIN_BARS or len(df_test) == 0:
            skipped += 1
            continue

        X_tr = df_train[ALL_FEATURES].values.astype('float32')
        y_tr = df_train['label_fh'].values
        X_te = df_test[ALL_FEATURES].values.astype('float32')

        m = lgb.LGBMClassifier(**WFO_PARAMS)
        m.fit(X_tr, y_tr)
        probs_window = m.predict_proba(X_te)[:, 1]

        for ts, p in zip(df_test.index, probs_window):
            all_probs[ts] = p

        if (i + 1) % 4 == 0 or i == len(windows) - 1:
            elapsed = time.perf_counter() - t0
            print(f'  [{scheme_key}] {i+1}/{len(windows)}  '
                  f'train={len(df_train):,}  test={len(df_test):,}  '
                  f'({elapsed:.0f}s)', flush=True)

    elapsed = time.perf_counter() - t0
    print(f'  [{scheme_key}] DONE — {len(all_probs):,} predictions  '
          f'skipped={skipped}  {elapsed:.1f}s')

    result = pd.Series(all_probs, name='prob').sort_index()
    result.to_frame().to_parquet(cache_path)
    return result


def fh_backtest(
    df_bt: pd.DataFrame,
    probs: np.ndarray,
    threshold: float,
    fee: float = 0.0,
) -> pd.DataFrame:
    """
    Fixed-horizon backtest with no-overlap constraint.

    Parameters
    ----------
    df_bt     : OOS dataframe with 'close' column
    probs     : P(TP) for each bar in df_bt
    threshold : minimum probability to enter a trade
    fee       : round-trip fee to deduct from pnl (0.0 = zero-fee, 0.0005 = 0.05%)

    Notes
    -----
    - All signals are LONG (Spot: maker entry, taker exit)
    - fee is applied as a flat deduction from raw pnl at exit
    - No funding cost for spot longs (SPOT_FUNDING_H = 0)
    """
    close_arr = df_bt['close'].values.astype('float64')
    index_arr = df_bt.index
    n         = len(df_bt)

    sig_idx = np.where(probs > threshold)[0]
    if len(sig_idx) == 0:
        return pd.DataFrame()

    trades    = []
    last_exit = -1

    for i in sig_idx:
        if i <= last_exit:
            continue
        dur      = min(FH_HORIZON, n - 1 - i)
        exit_bar = i + dur
        raw_pnl  = (close_arr[exit_bar] - close_arr[i]) / close_arr[i]
        pnl      = raw_pnl - fee   # flat fee deduction at exit
        last_exit = i + FH_HORIZON

        trades.append({
            'entry_time':    index_arr[i],
            'exit_time':     index_arr[exit_bar],
            'direction':     'long',
            'outcome':       'FH',
            'pnl_pct':       pnl,
            'pnl_pct_nofee': raw_pnl,
            'prob':          probs[i],
            'duration_bars': dur,
        })

    if not trades:
        return pd.DataFrame()
    return pd.DataFrame(trades).set_index('entry_time')

## Run All 4 WFO Schemes

~18 windows × 4 schemes = 72 fits. Expected: **~2 minutes** on Apple M-series.  
Results are cached — re-running the cell after a kernel restart is instant.

In [ ]:
wfo_results = {}
t_all = time.perf_counter()

for scheme_key, scheme in WFO_SCHEMES.items():
    print(f'\n{"="*60}')
    print(f'Scheme : {scheme["label"]}')
    print(f'{"="*60}')
    probs_series = run_wfo(scheme_key, scheme)
    wfo_results[scheme_key] = {'probs': probs_series, 'scheme': scheme}

print(f'\nAll schemes done in {(time.perf_counter()-t_all)/60:.1f} min')

## OOS Evaluation: AUC + Zero-Fee + Fee-Adjusted

For each scheme:
1. Compute OOS AUC on the 2024-11-10 → present window.
2. Run `fh_backtest(fee=0.0)` → zero-fee baseline.
3. Run `fh_backtest(fee=ROUND_TRIP_FEE)` → fee-adjusted (0.05% taker per trade).
4. Use scheme-specific optimal thresholds from v3 sensitivity analysis.

In [ ]:
import math

df_oos    = df_base[df_base.index >= OOS_START].copy()
y_oos     = df_oos['label_fh'].values
price_oos = df_oos[['close']]

print(f'OOS dataset : {len(df_oos):,} bars  '
      f'{df_oos.index[0].date()} -> {df_oos.index[-1].date()}')
print(f'OOS pos_rate: {y_oos.mean():.3f}')

# BTC price context
btc_start = price_oos['close'].iloc[0]
btc_end   = price_oos['close'].iloc[-1]
print(f'BTC start   : ${btc_start:,.0f}  |  BTC end: ${btc_end:,.0f}  '
      f'|  B&H: {(btc_end/btc_start - 1):+.2%}')
print()

summary_rows = []

def _kpis(trades, equity_arr):
    """Quick KPI dict from trades + equity array."""
    if trades.empty or len(equity_arr) == 0:
        return {}
    pnl    = trades['pnl_pct'].values.astype(float)
    n      = len(pnl)
    tot    = float(equity_arr[-1] - 1)
    pk     = np.maximum.accumulate(equity_arr)
    max_dd = float(((equity_arr - pk) / (pk + 1e-12)).min())
    wr     = float((pnl > 0).mean())
    ev     = float(pnl.mean())
    t0_t   = trades.index[0]; t1_t = trades.index[-1]
    n_days = max((t1_t - t0_t).total_seconds() / 86400, 1)
    years  = n_days / 365.25
    ann    = float(equity_arr[-1] ** (1.0 / max(years, 1/365)) - 1)
    tpd    = n / n_days
    sd     = float(pnl.std(ddof=1)) if n > 1 else 1e-9
    sharpe = float(pnl.mean() / sd * math.sqrt(tpd * 252)) if sd > 1e-12 else 0.0
    gp = pnl[pnl > 0].sum() if (pnl > 0).any() else 0.0
    gl = abs(pnl[pnl < 0].sum()) if (pnl < 0).any() else 1e-9
    return dict(n=n, tot=tot, ann=ann, sharpe=sharpe, max_dd=max_dd,
                wr=wr, ev=ev, pf=gp/gl)


for scheme_key, res in wfo_results.items():
    scheme    = res['scheme']
    thr       = SCHEME_THRESHOLDS[scheme_key]
    probs_s   = res['probs']

    probs_aligned = probs_s.reindex(df_oos.index).values.astype('float64')
    valid         = ~np.isnan(probs_aligned)

    auc = (roc_auc_score(y_oos[valid], probs_aligned[valid])
           if valid.sum() > 100 else float('nan'))

    probs_clean = np.nan_to_num(probs_aligned, nan=0.0)
    n_signals   = int((probs_clean > thr).sum())

    # Zero-fee backtest
    trades_nf = fh_backtest(df_oos, probs_clean, thr, fee=0.0)
    # Fee-adjusted backtest
    trades_f  = fh_backtest(df_oos, probs_clean, thr, fee=ROUND_TRIP_FEE)

    eq_nf = _equity_from_trades(trades_nf, df_oos.index, FH_HORIZON) if not trades_nf.empty else None
    eq_f  = _equity_from_trades(trades_f,  df_oos.index, FH_HORIZON) if not trades_f.empty  else None

    k_nf = _kpis(trades_nf, eq_nf.values) if eq_nf is not None else {}
    k_f  = _kpis(trades_f,  eq_f.values)  if eq_f  is not None else {}

    # Store
    res.update(dict(
        trades_nf=trades_nf, trades_f=trades_f,
        eq_nf=eq_nf,         eq_f=eq_f,
        kpis_nf=k_nf,        kpis_f=k_f,
        probs_aligned=probs_aligned,
        auc=auc, n_signals=n_signals, threshold=thr,
    ))

    summary_rows.append(dict(
        scheme=scheme_key, label=scheme['label'],
        threshold=thr, auc=auc, n_signals=n_signals,
        # zero-fee
        nf_trades=k_nf.get('n', 0),
        nf_ev_pct=k_nf.get('ev', float('nan')) * 100,
        nf_tot_pct=k_nf.get('tot', float('nan')) * 100,
        nf_sharpe=k_nf.get('sharpe', float('nan')),
        nf_maxdd_pct=k_nf.get('max_dd', float('nan')) * 100,
        nf_wr_pct=k_nf.get('wr', float('nan')) * 100,
        # fee-adjusted
        f_trades=k_f.get('n', 0),
        f_ev_pct=k_f.get('ev', float('nan')) * 100,
        f_tot_pct=k_f.get('tot', float('nan')) * 100,
        f_sharpe=k_f.get('sharpe', float('nan')),
        f_maxdd_pct=k_f.get('max_dd', float('nan')) * 100,
        f_wr_pct=k_f.get('wr', float('nan')) * 100,
    ))

    print(f'[{scheme_key:12s}] p>{thr}  AUC={auc:.5f}  '
          f'n_trades={k_nf.get("n",0):4d}  '
          f'EV_nofee={k_nf.get("ev",float("nan"))*100:+.4f}%  '
          f'EV_fee={k_f.get("ev",float("nan"))*100:+.4f}%')

summary_df = pd.DataFrame(summary_rows).set_index('scheme')
print()
print('Zero-fee vs Fee-adjusted summary:')
print(summary_df[['threshold', 'nf_ev_pct', 'f_ev_pct', 'nf_tot_pct', 'f_tot_pct',
                  'nf_sharpe', 'f_sharpe', 'f_maxdd_pct']].to_string())

## Monthly PnL Heatmap

Shows consistency of returns across calendar months.  
Green = month had positive aggregate PnL; Red = negative; White = no trades.

In [ ]:
# ── Style constants (mirror v8) ───────────────────────────────────────────────
ACCENT = '#F7931A'; BLUE = '#2962FF'; GREY = '#9E9E9E'
RED    = '#EF5350'; GREEN = '#26A69A'; PURPLE = '#7B1FA2'

mpl.rcParams.update({
    'font.family': 'serif', 'font.serif': ['DejaVu Serif'],
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'figure.dpi': 120,
})


def _monthly_pnl_matrix(trades_df: pd.DataFrame) -> pd.DataFrame:
    """Pivot table: rows = scheme + year, columns = month, values = total PnL %."""
    if trades_df.empty:
        return pd.DataFrame()
    t = trades_df.copy()
    t['year']  = t.index.year
    t['month'] = t.index.month
    pivot = t.groupby(['year', 'month'])['pnl_pct'].sum().unstack(fill_value=0) * 100
    # Ensure all 12 months present
    for m in range(1, 13):
        if m not in pivot.columns:
            pivot[m] = 0.0
    return pivot.sort_index(axis=1)


n_schemes = len(WFO_SCHEMES)
fig, axes = plt.subplots(n_schemes, 1, figsize=(14, 3.0 * n_schemes),
                         gridspec_kw={'hspace': 0.55})
if n_schemes == 1:
    axes = [axes]

fig.suptitle(
    'Monthly PnL Heatmap — Fee-adjusted (0.05% taker)\n'
    f'OOS {OOS_START.date()} → {df_oos.index[-1].date()}',
    fontweight='bold', fontsize=12,
)

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for ax, (sk, res) in zip(axes, wfo_results.items()):
    thr   = res['threshold']
    label = res['scheme']['label']
    pivot = _monthly_pnl_matrix(res['trades_f'])  # fee-adjusted

    if pivot.empty:
        ax.text(0.5, 0.5, f'{label}: no trades at p>{thr}',
                ha='center', va='center', transform=ax.transAxes, fontsize=11)
        ax.set_title(f'{label}  (p > {thr})', fontweight='bold', fontsize=10)
        continue

    vmax = max(abs(pivot.values.max()), abs(pivot.values.min()), 0.5)
    im = ax.imshow(
        pivot.values,
        cmap='RdYlGn',
        vmin=-vmax, vmax=vmax,
        aspect='auto',
    )

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([month_names[m - 1] for m in pivot.columns], fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([str(y) for y in pivot.index], fontsize=8)

    # Annotate cells with PnL value
    for r, year in enumerate(pivot.index):
        for c, month in enumerate(pivot.columns):
            v = pivot.loc[year, month]
            txt_color = 'white' if abs(v) > vmax * 0.6 else 'black'
            ax.text(c, r, f'{v:+.2f}%', ha='center', va='center',
                    fontsize=7.5, color=txt_color, fontweight='bold')

    n_pos = int((pivot.values > 0).sum())
    n_neg = int((pivot.values < 0).sum())
    n_zero= int((pivot.values == 0).sum())
    ax.set_title(
        f'{label}  (p > {thr})  —  '
        f'Monthly cells: {n_pos} positive / {n_neg} negative / {n_zero} no-trade',
        fontweight='bold', fontsize=10,
    )
    fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label='PnL %')

plt.savefig(ARTIFACTS_DIR / '00_monthly_pnl_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved -> {ARTIFACTS_DIR}/00_monthly_pnl_heatmap.png')

## Multi-Scheme Comparison: Zero-fee vs Fee-adjusted

In [ ]:
scheme_keys   = list(WFO_SCHEMES.keys())
scheme_labels = [WFO_SCHEMES[k]['label'] for k in scheme_keys]
palette       = [ACCENT, BLUE, GREEN, PURPLE]
x             = np.arange(len(scheme_keys))
bar_w         = 0.36

def _safe(col, default=0.0):
    return [float(summary_df.loc[k, col])
            if k in summary_df.index and not pd.isna(summary_df.loc[k, col])
            else default
            for k in scheme_keys]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    'WFO Scheme Comparison — Fixed Horizon (>0.3% / 6h)\n'
    f'OOS {OOS_START.date()} → {df_oos.index[-1].date()}  |  '
    f'Per-scheme optimal thresholds  |  B&H ≈ 0%',
    fontweight='bold', fontsize=12,
)

def _label_bars(ax, bars, vals, fmt='{:.4f}', dy=None):
    yrange = ax.get_ylim()
    dy = dy or (yrange[1] - yrange[0]) * 0.02
    for bar, v in zip(bars, vals):
        vy = v + dy if v >= 0 else v - dy * 3
        ax.text(bar.get_x() + bar.get_width() / 2, vy,
                fmt.format(v), ha='center',
                va='bottom' if v >= 0 else 'top', fontsize=8.5, fontweight='bold')

tick_lbl = [f'{WFO_SCHEMES[k]["label"][:16]}\n(p>{SCHEME_THRESHOLDS[k]})'
            for k in scheme_keys]

# 1. OOS AUC
ax = axes[0, 0]
vals = _safe('auc', 0.5)
bars = ax.bar(x, vals, color=palette, alpha=0.85, edgecolor='white')
ax.axhline(0.50, color=GREY, ls='--', lw=1.0, label='Random (0.50)')
ax.set_xticks(x); ax.set_xticklabels(tick_lbl, fontsize=8)
ax.set_title('OOS AUC', fontweight='bold')
ax.set_ylim(0.47, max(vals) + 0.03)
_label_bars(ax, bars, vals, '{:.4f}', (max(vals)-0.47)*0.04)
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.25)

# 2. EV per trade: zero-fee vs fee-adj
ax = axes[0, 1]
nf_ev = _safe('nf_ev_pct')
f_ev  = _safe('f_ev_pct')
b1 = ax.bar(x - bar_w/2, nf_ev, bar_w, color=ACCENT, alpha=0.85, label='Zero-fee', edgecolor='white')
b2 = ax.bar(x + bar_w/2, f_ev,  bar_w, color=GREEN,  alpha=0.85, label='Fee-adj (0.05%)', edgecolor='white')
ax.axhline(0,    color=GREY, lw=1.0)
ax.axhline(0.05, color=RED,   ls='--', lw=1.0, label='Fee hurdle 0.05%')
ax.axhline(0.40, color=GREEN, ls='--', lw=1.0, label='EV target 0.40%')
_label_bars(ax, b1, nf_ev, '{:+.3f}%')
_label_bars(ax, b2, f_ev,  '{:+.3f}%')
ax.set_xticks(x); ax.set_xticklabels(tick_lbl, fontsize=8)
ax.set_title('EV per Trade %', fontweight='bold')
ax.legend(fontsize=7.5, ncol=2); ax.grid(axis='y', alpha=0.25)

# 3. Total return: zero-fee vs fee-adj
ax = axes[0, 2]
nf_tot = _safe('nf_tot_pct')
f_tot  = _safe('f_tot_pct')
b1 = ax.bar(x - bar_w/2, nf_tot, bar_w, color=ACCENT, alpha=0.85, label='Zero-fee', edgecolor='white')
b2 = ax.bar(x + bar_w/2, f_tot,  bar_w, color=GREEN,  alpha=0.85, label='Fee-adj', edgecolor='white')
ax.axhline(0, color=GREY, lw=1.0)
_label_bars(ax, b1, nf_tot, '{:+.1f}%')
_label_bars(ax, b2, f_tot,  '{:+.1f}%')
ax.set_xticks(x); ax.set_xticklabels(tick_lbl, fontsize=8)
ax.set_title('Total Return %', fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.25)

# 4. Sharpe: zero-fee vs fee-adj
ax = axes[1, 0]
nf_sh = _safe('nf_sharpe')
f_sh  = _safe('f_sharpe')
b1 = ax.bar(x - bar_w/2, nf_sh, bar_w, color=ACCENT, alpha=0.85, label='Zero-fee', edgecolor='white')
b2 = ax.bar(x + bar_w/2, f_sh,  bar_w, color=GREEN,  alpha=0.85, label='Fee-adj', edgecolor='white')
ax.axhline(0, color=GREY, lw=1.0)
_label_bars(ax, b1, nf_sh, '{:.3f}')
_label_bars(ax, b2, f_sh,  '{:.3f}')
ax.set_xticks(x); ax.set_xticklabels(tick_lbl, fontsize=8)
ax.set_title('Sharpe (ann., trade-level)', fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.25)

# 5. Win rate: zero-fee vs fee-adj
ax = axes[1, 1]
nf_wr = _safe('nf_wr_pct')
f_wr  = _safe('f_wr_pct')
b1 = ax.bar(x - bar_w/2, nf_wr, bar_w, color=ACCENT, alpha=0.85, label='Zero-fee', edgecolor='white')
b2 = ax.bar(x + bar_w/2, f_wr,  bar_w, color=GREEN,  alpha=0.85, label='Fee-adj', edgecolor='white')
ax.axhline(50, color=GREY, ls='--', lw=1.0, label='50% breakeven')
_label_bars(ax, b1, nf_wr, '{:.1f}%')
_label_bars(ax, b2, f_wr,  '{:.1f}%')
ax.set_xticks(x); ax.set_xticklabels(tick_lbl, fontsize=8)
ax.set_title('Win Rate %', fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.25)

# 6. Max drawdown: zero-fee vs fee-adj
ax = axes[1, 2]
nf_dd = _safe('nf_maxdd_pct')
f_dd  = _safe('f_maxdd_pct')
b1 = ax.bar(x - bar_w/2, nf_dd, bar_w, color=ACCENT, alpha=0.70, label='Zero-fee', edgecolor='white')
b2 = ax.bar(x + bar_w/2, f_dd,  bar_w, color=RED,    alpha=0.70, label='Fee-adj', edgecolor='white')
_label_bars(ax, b1, nf_dd, '{:.1f}%')
_label_bars(ax, b2, f_dd,  '{:.1f}%')
ax.set_xticks(x); ax.set_xticklabels(tick_lbl, fontsize=8)
ax.set_title('Max Drawdown %', fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / '01_scheme_comparison_zero_vs_fee.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved -> {ARTIFACTS_DIR}/01_scheme_comparison_zero_vs_fee.png')

## Equity Curves Overlay — All Schemes, Both Fee Regimes

In [ ]:
fig = plt.figure(figsize=(14, 11))
gs  = fig.add_gridspec(3, 1, height_ratios=[3.0, 1.2, 1.2], hspace=0.08)
ax_eq = fig.add_subplot(gs[0])
ax_dd = fig.add_subplot(gs[1], sharex=ax_eq)
ax_sp = fig.add_subplot(gs[2], sharex=ax_eq)

fig.suptitle(
    f'Equity Curves — All Schemes, Zero-fee vs Fee-adjusted\n'
    f'OOS {OOS_START.date()} → {df_oos.index[-1].date()}  |  B&H ≈ 0%',
    fontweight='bold', fontsize=11,
)

bh_ts  = price_oos['close'] / price_oos['close'].iloc[0]
bh_pct = (bh_ts.values - 1) * 100
bh_pk  = np.maximum.accumulate(bh_ts.values)
bh_dd  = (bh_ts.values - bh_pk) / (bh_pk + 1e-12)

ax_eq.plot(price_oos.index, bh_pct,
           color=BLUE, lw=1.2, ls='--', alpha=0.70,
           label=f'Buy & Hold  ({bh_ts.values[-1]-1:+.2%})')
ax_dd.fill_between(price_oos.index, bh_dd * 100, 0,
                   color=BLUE, alpha=0.20, label='B&H DD')

for (sk, res), color in zip(wfo_results.items(), palette):
    lbl    = res['scheme']['label'][:20]
    thr    = res['threshold']
    t_nf   = res['trades_nf']
    t_f    = res['trades_f']

    # Zero-fee: solid
    if not t_nf.empty:
        eq_nf  = _equity_from_trades(t_nf, price_oos.index, FH_HORIZON)
        eq_pct = (eq_nf.values - 1) * 100
        pk     = np.maximum.accumulate(eq_nf.values)
        dd     = (eq_nf.values - pk) / (pk + 1e-12)
        n_t    = len(t_nf)
        ax_eq.plot(price_oos.index, eq_pct, color=color, lw=1.6,
                   label=f'{lbl} 0-fee ({eq_pct[-1]:+.1f}%  n={n_t})')
        ax_dd.fill_between(price_oos.index, dd * 100, 0, color=color, alpha=0.35)
        ax_dd.plot(price_oos.index, dd * 100, color=color, lw=0.6)

    # Fee-adj: dashed
    if not t_f.empty:
        eq_f   = _equity_from_trades(t_f, price_oos.index, FH_HORIZON)
        eq_pct = (eq_f.values - 1) * 100
        n_t    = len(t_f)
        ax_eq.plot(price_oos.index, eq_pct, color=color, lw=1.1, ls='--', alpha=0.75,
                   label=f'{lbl} fee-adj ({eq_pct[-1]:+.1f}%)')

    # Fee drag fill for the largest (3-month) scheme
    if sk == 'short_3mo' and not t_nf.empty and not t_f.empty:
        eq_nf_v = _equity_from_trades(t_nf, price_oos.index, FH_HORIZON).values
        eq_f_v  = _equity_from_trades(t_f,  price_oos.index, FH_HORIZON).values
        ax_eq.fill_between(price_oos.index,
                           (eq_f_v - 1) * 100, (eq_nf_v - 1) * 100,
                           color=RED, alpha=0.08, label='3-mo fee drag')

ax_eq.axhline(0, color=GREY, lw=0.7, ls=':')
ax_eq.set_ylabel('Cumulative Return %')
ax_eq.legend(ncol=2, fontsize=7.5); ax_eq.grid(axis='y', alpha=0.25)
plt.setp(ax_eq.xaxis.get_majorticklabels(), visible=False)

ax_dd.axhline(0, color=GREY, lw=0.7)
ax_dd.set_ylabel('Drawdown %')
ax_dd.legend(ncol=3, fontsize=7); ax_dd.grid(axis='y', alpha=0.25)
plt.setp(ax_dd.xaxis.get_majorticklabels(), visible=False)

# ── Bottom: BTC price (context for regime) ──────────────────────────────────
ax_sp.plot(price_oos.index, price_oos['close'].values / 1000,
           color=ACCENT, lw=1.0, alpha=0.85)
ax_sp.fill_between(price_oos.index,
                   price_oos['close'].values / 1000, 0,
                   color=ACCENT, alpha=0.08)
ax_sp.set_ylabel('BTC Price (k$)')
ax_sp.set_title('BTC/USDT 5m — OOS Price Reference', fontsize=9)
ax_sp.grid(axis='y', alpha=0.20)
ax_sp.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 3, 5, 7, 9, 11]))
ax_sp.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax_sp.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=8)

plt.savefig(ARTIFACTS_DIR / '02_equity_overlay.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved -> {ARTIFACTS_DIR}/02_equity_overlay.png')

## Individual Tearsheets (schemes with ≥ 10 trades)

Zero-fee tearsheets only — fee-adjusted comparison in next section.

In [ ]:
MIN_TRADES_TEARSHEET = 10

for scheme_key, res in wfo_results.items():
    trades_nf = res['trades_nf']
    auc       = res['auc']
    label     = res['scheme']['label']
    thr       = res['threshold']
    n_signals = res['n_signals']

    if len(trades_nf) < MIN_TRADES_TEARSHEET:
        print(f'[{scheme_key}] Skipping tearsheet: only {len(trades_nf)} trades '
              f'(< {MIN_TRADES_TEARSHEET})')
        continue

    params_d = dict(
        wfo_scheme      = label,
        fh_horizon_bars = FH_HORIZON,
        fh_threshold    = FH_THRESHOLD,
        prob_threshold  = thr,
        embargo_bars    = EMBARGO_BARS,
        n_signals       = n_signals,
        n_trades        = len(trades_nf),
        n_muted         = 0,
    )
    metrics_d = dict(
        auc        = auc,
        oos_return = res['kpis_nf'].get('tot', float('nan')),
        oos_sharpe = res['kpis_nf'].get('sharpe', float('nan')),
    )

    title = (
        f'WFO Tearsheet — {label}  (Zero-fee)\n'
        f'FH >0.3% / 6h  |  OOS {OOS_START.date()}–{df_oos.index[-1].date()}  |  '
        f'p > {thr}  |  AUC = {auc:.5f}'
    )

    save_path = ARTIFACTS_DIR / f'tearsheet_{scheme_key}_nofee.png'
    fig = generate_tearsheet(
        trades_df   = trades_nf,
        price_df    = price_oos,
        prob_series = pd.Series(res['probs_aligned'], index=df_oos.index),
        params      = params_d,
        metrics     = metrics_d,
        title       = title,
        fig_size    = (14, 17),
        save_path   = save_path,
    )
    plt.show()
    print(f'Saved -> {save_path}')

## Per-Scheme: Zero-fee vs Fee-adjusted Comparison Tearsheets

In [ ]:
for scheme_key, res in wfo_results.items():
    trades_nf = res['trades_nf']
    trades_f  = res['trades_f']
    label     = res['scheme']['label']
    thr       = res['threshold']

    if len(trades_nf) < MIN_TRADES_TEARSHEET:
        print(f'[{scheme_key}] Skipping fee-comparison: only {len(trades_nf)} trades')
        continue

    params_d = dict(
        wfo_scheme      = label,
        fh_horizon_bars = FH_HORIZON,
        fh_threshold    = FH_THRESHOLD,
        prob_threshold  = thr,
        embargo_bars    = EMBARGO_BARS,
    )

    save_path = ARTIFACTS_DIR / f'fee_comparison_{scheme_key}.png'
    fig = generate_fee_comparison(
        trades_nofee = trades_nf,
        trades_fee   = trades_f,
        price_df     = price_oos,
        scheme_label = label,
        fee_label    = f'Spot long: maker 0% + taker {SPOT_TAKER_FEE:.2%} = {ROUND_TRIP_FEE:.2%} rt',
        params       = params_d,
        fig_size     = (14, 10),
        save_path    = save_path,
    )
    plt.show()
    print(f'Saved -> {save_path}')

## Threshold Sensitivity (fee-adjusted)

Validates the per-scheme threshold choices against the shorter 2024-11-10 OOS window.

In [ ]:
thresholds = [0.55, 0.60, 0.65, 0.70, 0.72, 0.75, 0.78, 0.80, 0.85]

print('Fee-adjusted threshold sensitivity (0.05% taker per trade):')
print(f'{"Threshold":>12}  '
      + '  '.join(f'{WFO_SCHEMES[k]["label"][:20]:>22}' for k in scheme_keys))
print('-' * (14 + len(scheme_keys) * 24))

for thresh in thresholds:
    parts = [f'  p > {thresh:.2f}   ']
    for sk in scheme_keys:
        probs = np.nan_to_num(wfo_results[sk]['probs_aligned'], nan=0.0)
        t     = fh_backtest(df_oos, probs, thresh, fee=ROUND_TRIP_FEE)
        if t.empty:
            parts.append(f'{"-- / -- / --":>24}')
        else:
            ev  = t['pnl_pct'].mean() * 100
            n   = len(t)
            wr  = (t['pnl_pct'] > 0).mean() * 100
            marker = '*' if thresh == SCHEME_THRESHOLDS[sk] else ' '
            parts.append(f'  {ev:+.3f}% n={n:<3d} wr={wr:.0f}%{marker}')
    print(''.join(parts))

print()
print('* = selected threshold for this scheme')

In [ ]:
# Save summary CSV
csv_path = ARTIFACTS_DIR / 'wfo_summary_v4.csv'
summary_df.to_csv(csv_path)
print(f'Summary CSV -> {csv_path}')
print()
print(f'Artifacts in {ARTIFACTS_DIR}:')
for p in sorted(ARTIFACTS_DIR.iterdir()):
    print(f'  {p.name:<60}  {p.stat().st_size / 1024:>8.1f} KB')

## Conclusion

### True-alpha window (B&H ≈ 0%)

By starting the OOS at 2024-11-10 (BTC ≈ \$73k, same as today), any positive return
cannot be attributed to Bitcoin's secular uptrend — it must come from structural signal.

### Fee survival gate

Round-trip cost = 0.05% (maker 0% entry + taker 0.05% exit on Spot longs).  
A scheme is viable only if fee-adjusted EV > 0.05%.

| Scheme | Zero-fee EV | Fee-adj EV | Viable? |
|--------|------------|------------|--------|
| Expanding | depends | depends | Low-sample — monitor |
| 2-Year | should be high | check | Likely yes |
| 1-Year | should be high | check | Likely yes |
| 3-Month | ~0.175% | ~0.125% | Borderline — many trades offset cost |

### Next steps

| Condition | Action |
|-----------|--------|
| Fee-adj EV > 0.40% for ≥ 1 scheme | `04_lgbm_fees_wfo.ipynb` — full fee model with slippage |
| Fee-adj EV 0.10–0.40% | `05_lgbm_mamba_stack.ipynb` — Mamba P(TP) as 44th meta-feature |
| All schemes below 0.10% | Switch to 1h base timeframe or add order-book features |

### Architecture note on short signals

The current FH label is long-only (`fwd_ret > +0.3%`).  
Adding a symmetric short label (`fwd_ret < -0.3%`) would route signals to Futures where
funding is **received** (+0.00077%/h), further improving fee-adjusted returns for bear regimes.